In [ ]:
import whisperx

device = "cpu"
audio_file = "./sample/1. DANG VU KHOA 2813917876.wav"
batch_size = 16 # reduce if low on GPU mem
compute_type = "float32" # change to "int8" if low on GPU mem (may reduce accuracy)

# 1. Transcribe with original whisper (batched)
model = whisperx.load_model("large-v3", device, compute_type=compute_type)

# save model to local path (optional)
# model_dir = "/path/"
# model = whisperx.load_model("large-v2", device, compute_type=compute_type, download_root=model_dir)

audio = whisperx.load_audio(audio_file)
result = model.transcribe(audio, batch_size=batch_size)
print(result["segments"]) # before alignment


/Users/mac/Programming/Python/source/quantumcomputing/LearnAI/asr/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/mac/Programming/Python/source/quantumcomputing/LearnAI/asr/.venv/lib/python3.11/site-packages/pyannote/audio/core/io.py:212: UserWarning: torchaudio._backend.list_audio_backends has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be removed from the 2.9 release. 
  torchaudio.list_audio_backends()
/Users/mac/Programming/Python/source/quantumcomputing/LearnAI/asr/.venv/lib/python3.11/site-packages/speechbrain

In [3]:
# 2. Align whisper output
model_a, metadata = whisperx.load_align_model(language_code=result["language"], device=device)
result = whisperx.align(result["segments"], model_a, metadata, audio, device, return_char_alignments=False)

print(result["segments"]) # after alignment

preprocessor_config.json:   0%|          | 0.00/256 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/397 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/30.0 [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/380M [00:00<?, ?B/s]

Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at nguyenvulebinh/wav2vec2-base-vi and are newly initialized: ['lm_head.bias', 'lm_head.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[{'start': 0.773, 'end': 30.375, 'text': ' Điểm thứ 2 mà chúng tôi cũng xin thưa với quý vị là cũng trong cái buổi nói chuyện đó thì tôi có đề cập tới một phụ nữ Việt Nam đã sống tại Mỹ 17 năm mà bị trục xuất thì thưa quý vị bản tin này chúng tôi cũng đọc ở trên báo hoặc là trên mạng nhưng mà không có kiểm chứng chi tiết cho chính xác trước khi lên đường qua Âu Chộc thì chúng tôi xin rút lại câu chuyện này để tránh sự hiểu lầm về chính sách di trú của Hoa Kỳ Thưa quý vị', 'words': [{'word': 'Điểm', 'start': np.float64(0.773), 'end': np.float64(0.853), 'score': np.float64(0.01)}, {'word': 'thứ', 'start': np.float64(0.873), 'end': np.float64(0.933), 'score': np.float64(0.011)}, {'word': '2', 'start': np.float64(0.953), 'end': np.float64(1.134), 'score': np.float64(0.011)}, {'word': 'mà', 'start': np.float64(1.154), 'end': np.float64(1.234), 'score': np.float64(0.01)}, {'word': 'chúng', 'start': np.float64(1.254), 'end': np.float64(1.354), 'score': np.float64(0.011)}, {'word': 'tôi', 'sta

model.safetensors:   0%|          | 0.00/380M [00:00<?, ?B/s]

In [4]:
# 3. Assign speaker labels
diarize_model = whisperx.diarize.DiarizationPipeline(use_auth_token=YOUR_HF_TOKEN, device=device)

# add min/max number of speakers if known
diarize_segments = diarize_model(audio)
# diarize_model(audio, min_speakers=min_speakers, max_speakers=max_speakers)

result = whisperx.assign_word_speakers(diarize_segments, result)
print(diarize_segments)
print(result["segments"]) # segments are now assigned speaker IDs


NameError: name 'YOUR_HF_TOKEN' is not defined